# Reconstruction de la table HAR (six configurations)

Les deux premières cellules chargent automatiquement le dernier lot produit par `experiments/scripts/run_HAR_table8.py`. Le lot contient les modèles **feedforward** et **récurrents**, chacun avec des délais **axonaux**, **hybrides** ou **synaptiques**, pour les seeds 0, 1 et 2.

La table est reconstruite uniquement à partir des `final_test.json` des runs terminés et utilise l'écart-type population (`ddof=0`), comme dans le tableau du support. 



In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'experiments/train.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError("Racine du dépôt introuvable.")

# Mettre ici un Path précis pour analyser un ancien lot. None choisit le plus récent.
RUN_ROOT = None
TABLE8_ROOT = ROOT / 'exp' / 'HAR' / 'table8'
if RUN_ROOT is None:
    candidates = sorted(path for path in TABLE8_ROOT.glob('*') if path.is_dir())
    if not candidates:
        raise FileNotFoundError(
            f"Aucun lot dans {TABLE8_ROOT}. Lancer d'abord : "
            "python experiments/scripts/run_HAR_table8.py"
        )
    RUN_ROOT = candidates[-1]

EXPORT = ROOT / 'figures' / 'har_comparison'
EXPORT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .2})
MODELS = {
    'SNN_axonal_feedforward_delays': ('Feedforward', 'Axonal', 58_037),
    'SNN_hybrid_feedforward_delays': ('Feedforward', 'Hybride', 58_037),
    'SNN_synaptic_feedforward_delays': ('Feedforward', 'Synaptique', 171_666),
    'SNN_axonal_recurrent_delays': ('Récurrent', 'Axonal', 105_522),
    'SNN_recurrent_hybrid_delays': ('Récurrent', 'Hybride', 105_522),
    'SNN_synaptic_recurrent_delays': ('Récurrent', 'Synaptique', 152_578),
}

def read_curve(path):
    if not path.exists():
        return pd.DataFrame(columns=['epoch', 'acc', 'loss'])
    raw = pd.read_csv(path, index_col=0)
    if raw.shape[0] != 2:
        raise ValueError(f"Format inattendu : {path}")
    return pd.DataFrame({'epoch': pd.to_numeric(raw.columns) + 1,
                         'acc': raw.iloc[0].to_numpy(dtype=float),
                         'loss': raw.iloc[1].to_numpy(dtype=float)}).sort_values('epoch')

runs = []
for model, (architecture, family, parameters) in MODELS.items():
    for path in sorted((RUN_ROOT / model).glob('seed*/config.json')):
        folder = path.parent
        cfg = json.loads(path.read_text(encoding='utf-8'))
        maximum = (
            cfg.get('hybrid_offset_scale', cfg.get('hybrid_max_synaptic_delay'))
            if family == 'Hybride' else None
        )
        group = f'{architecture} — {family}'
        final_path = folder / 'final_test.json'
        final = json.loads(final_path.read_text(encoding='utf-8')) if final_path.exists() else {}
        runs.append(dict(path=folder, config=cfg, final=final, model=model,
                         architecture=architecture, family=family, group=group,
                         parameters=parameters, maximum=maximum, seed=cfg.get('seed'),
                         name=folder.name,
                         train=read_curve(folder / 'train_res.csv'),
                         val=read_curve(folder / 'val_res.csv')))
if not runs:
    raise ValueError(f'Aucun run trouvé dans {RUN_ROOT}.')
groups = list(dict.fromkeys(r['group'] for r in runs))
palette = plt.get_cmap('tab10').colors
colors = {group: palette[index % len(palette)] for index, group in enumerate(groups)}
def label(r):
    return f"{r['group']} | seed {r['seed']}"
def savefig(name):
    plt.savefig(EXPORT / f'{name}.png', dpi=180, bbox_inches='tight')
    plt.show()

summary = pd.DataFrame([{'architecture': r['architecture'], 'délais': r['family'],
    'modèle': r['model'], 'paramètres': r['parameters'], 'seed': r['seed'],
    'run': r['name'], 'époques observées': len(r['train']), 'terminé': bool(r['final']),
    'acc finale (%)': r['final'].get('acc', np.nan),
    'loss finale': r['final'].get('loss', np.nan),
    'meilleure époque (1-based)': r['final'].get('best_epoch', np.nan) + 1,
    'offset seed': r['config'].get('hybrid_delay_seed') if r['family'] == 'Hybride' else None
} for r in runs])
print(f'Lot analysé : {RUN_ROOT}')
display(summary)
summary.to_csv(EXPORT / 'runs.csv', index=False)

# Table de la diapositive 6 / Table 8 du rapport.
completed = [r for r in runs if r['final'].get('acc') is not None and len(r['train']) >= 100]
table_source = pd.DataFrame([{
    'architecture': r['architecture'], 'délais': r['family'],
    'paramètres': r['parameters'], 'seed': r['seed'],
    'accuracy': float(r['final']['acc'])
} for r in completed])

if table_source.empty:
    print('Aucun run complet de 100 époques : la table sera disponible après les entraînements.')
    table6 = pd.DataFrame()
else:
    table6_raw = (table_source.groupby(['architecture', 'délais', 'paramètres'], sort=False)
        .agg(n=('accuracy', 'size'), moyenne=('accuracy', 'mean'),
             écart_type=('accuracy', lambda values: values.std(ddof=0)))
        .reset_index())
    architecture_order = {'Feedforward': 0, 'Récurrent': 1}
    delay_order = {'Axonal': 0, 'Hybride': 1, 'Synaptique': 2}
    table6_raw['ordre_architecture'] = table6_raw['architecture'].map(architecture_order)
    table6_raw['ordre_délais'] = table6_raw['délais'].map(delay_order)
    table6_raw = table6_raw.sort_values(['ordre_architecture', 'ordre_délais'])
    table6_raw['Accuracy (%)'] = table6_raw.apply(
        lambda row: f"{row['moyenne']:.3f} ± {row['écart_type']:.3f}"
        + ('' if row['n'] == 3 else f" ({int(row['n'])}/3 runs)"), axis=1)
    table6 = table6_raw[['architecture', 'délais', 'paramètres', 'Accuracy (%)']]
    display(table6.style.hide(axis='index').format({'paramètres': '{:,.0f}'}))
    table6.to_csv(EXPORT / 'table6_HAR.csv', index=False)
